# PD03 · Integrar EVA con DIVIPOLA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es comprobar una unión territorial y calcular un rendimiento con el denominador adecuado.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD03_integracion_territorial.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD03_integracion_territorial.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 7, 9, 10 y la receta de razón agregada de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD03"
archivos = ['eva.csv', 'divipola.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)
divipola = pd.read_csv(RAW / "divipola.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03


## 1. Dimensión territorial · 10 minutos

Crear `codigo_municipio` desde `Código Municipio` de DIVIPOLA. Comprobar cinco cifras y unicidad. Contar los valores de `Tipo: Municipio / Isla / Área no municipalizada`. Construir `dimension` con código, nombre departamental, nombre municipal y clasificación territorial.

📘 **Apoyo:** `.str.strip`, `.str.fullmatch(r"[0-9]{5}")`, `is_unique` y `value_counts`.

✅ **Comprobación:** La dimensión conserva las 1.122 entidades y sus tres tipos. La columna de código es única y no contiene vacíos.

In [2]:
# Desarrollo de la actividad 1
# Preparar dimension y mostrar la distribución por tipo.
# Comprobar formato y unicidad del código.
# Crear código de municipio desde DIVIPOLA
import pandas as pd

ds_divipola = pd.read_csv('../kit/data/raw/divipola.csv')
#display(ds_divipola)
print("------------------")
# Crear codigo_municipio
ds_divipola["codigo_municipio"] = (
    ds_divipola["Código Municipio"]
    .astype(str)
    .str.strip()
    .str.zfill(5)
)
#print(ds_divipola["codigo_municipio"].head())
#print("--------------")
# Comprobar cinco cifras
assert ds_divipola["codigo_municipio"].str.fullmatch(r"[0-9]{5}").all()

# Comprobar que no haya vacíos
assert ds_divipola["codigo_municipio"].notna().all()
assert (ds_divipola["codigo_municipio"] != "").all()

# Comprobar unicidad
assert ds_divipola["codigo_municipio"].is_unique

# Contar tipos de entidad
tipos = ds_divipola[
    "Tipo: Municipio / Isla / Área no municipalizada"
].value_counts()

# Comprobaciones
assert len(ds_divipola) == 1122

assert tipos.to_dict() == {
    "Municipio": 1103,
    "Área no municipalizada": 18,
    "Isla": 1,
}

# Construir dimension
dimension = ds_divipola[
    [
        "codigo_municipio",
        "Nombre Departamento",
        "Nombre Municipio",
        "Tipo: Municipio / Isla / Área no municipalizada"
    ]
].copy()

# Renombrar columnas
dimension = dimension.rename(columns={
    "Nombre Departamento": "nombre_departamental",
    "Nombre Municipio": "nombre_municipal",
    "Tipo: Municipio / Isla / Área no municipalizada":
        "clasificacion_territorial"
})

# Comprobaciones finales
assert len(dimension) == 1122
assert dimension["codigo_municipio"].is_unique
assert dimension["codigo_municipio"].notna().all()
assert dimension["codigo_municipio"].str.fullmatch(r"[0-9]{5}").all()

assert dimension["clasificacion_territorial"].value_counts().to_dict() == {
    "Municipio": 1103,
    "Área no municipalizada": 18,
    "Isla": 1,
}

print(tipos)
print("-------------------")
display(dimension.head())


------------------
Tipo: Municipio / Isla / Área no municipalizada
Municipio                 1103
Área no municipalizada      18
Isla                         1
Name: count, dtype: int64
-------------------


,codigo_municipio,nombre_departamental,nombre_municipal,clasificacion_territorial
0,05001,ANTIOQUIA,MEDELLÍN,Municipio
1,05002,ANTIOQUIA,ABEJORRAL,Municipio
2,05004,ANTIOQUIA,ABRIAQUÍ,Municipio
3,05021,ANTIOQUIA,ALEJANDRÍA,Municipio
4,05030,ANTIOQUIA,AMAGÁ,Municipio


**Interpretación del resultado:**

_Se creo codigo_municipio de DIVIPOLA, luego se hizo la comprobacion de cinco cifras y su unicidad.Posteriormente, se contaron los valores deTipo: Municipio / Isla / Área no municipalizada, para despues construir dimension con las columnas código, nombre departamental, nombre municipal y clasificación territorial, como se ve anteriormemte._

_Tambien  se comprobo que se conservaron los 1122 entidades y sus tres tipos. Ademas, se tuvo que añadir un cero a la izquierda de cada codigo_municipo que solo tuviera 4 cifras porque se necesitaban que todos los valores de codigo_municipio tuvieran 5 cifras o digitos._ 

## 2. Códigos EVA sin correspondencia · 15 minutos

Preparar `codigo_municipio` desde `Código Dane municipio` y `anio` desde `Año` de EVA. Conservar originales; los códigos vacíos o mal formados deben quedar identificados para revisión. Completar ceros solo en textos numéricos válidos de menor longitud. Crear combinaciones distintas de código y año y realizar un anti-join frente a dimension. Guardar el resultado en `sin_correspondencia`.

📘 **Apoyo:** `.str.fullmatch`, `.where`, `.str.zfill`, `drop_duplicates` y `merge(how="left", indicator=True, validate="many_to_one")`.

✅ **Comprobación:** El código municipal es la clave de unión. El año se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA. El listado se muestra aunque quede vacío.

In [3]:
# Desarrollo de la actividad 2
# Preparar código y año en eva.
# Obtener sin_correspondencia con código municipal y año.
import pandas as pd

# Leer EVA
eva = pd.read_csv("../kit/data/raw/eva.csv")
display(eva)
# Conservar originales
eva["codigo_municipio_original"] = eva["Código Dane municipio"]
eva["anio_original"] = eva["Año"]

# Convertir código a texto y quitar espacios
codigo = (
    eva["Código Dane municipio"]
    .astype("string")
    .str.strip()
)

# Identificar códigos que son numéricos y tienen entre 1 y 5 cifras
codigo_valido = codigo.str.fullmatch(r"[0-9]{1,5}")

# Crear codigo_municipio:
# solo se completa con ceros cuando el código es numérico válido
eva["codigo_municipio"] = (
    codigo
    .where(codigo_valido)
    .str.zfill(5)
)

# Crear año
eva["anio"] = eva["Año"]

# Crear combinaciones distintas de código y año
combinaciones = eva[
    ["codigo_municipio", "anio"]
].drop_duplicates()

# Anti-join contra dimension
sin_correspondencia = (
    combinaciones
    .merge(
        dimension[["codigo_municipio"]],
        on="codigo_municipio",
        how="left",
        indicator=True,
        validate="many_to_one"
    )
)

# Conservar únicamente los códigos que no existen en dimension
sin_correspondencia = sin_correspondencia[
    sin_correspondencia["_merge"] == "left_only"
].drop(columns="_merge")

print("----------------------")
# Mostrar resultado aunque esté vacío
display(sin_correspondencia)

,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Grupo cultivo,Subgrupo,Cultivo,Desagregación cultivo,Año,Periodo,Área sembrada,Área cosechada,Producción,Rendimiento,Ciclo del cultivo,Estado físico del cultivo,Código del cultivo,Nombre científico del cultivo
0,5,Antioquia,5001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021A,8.0,8.0,128.00,16.00,Transitorio,En fresco,1050600,Apium graveolens
1,5,Antioquia,5001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021B,8.0,8.0,152.00,19.00,Transitorio,En fresco,1050600,Apium graveolens
2,5,Antioquia,5001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022A,8.0,8.0,127.44,15.93,Transitorio,En fresco,1050600,Apium graveolens
3,5,Antioquia,5001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022B,8.0,8.0,152.00,19.00,Transitorio,En fresco,1050600,Apium graveolens
4,5,Antioquia,5001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2023,2023A,8.0,7.0,133.00,19.00,Transitorio,En fresco,1050600,Apium graveolens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166727,99,Vichada,99773,Cumaribo,Raíces y tubérculos,Raíces y tubérculos,Yuca,Yuca consumo en fresco,2021,2021A,380.0,380.0,3420.00,9.00,Transitorio,En fresco,1081001,Manihot esculenta
166728,99,Vichada,99773,Cumaribo,Raíces y tubérculos,Raíces y tubérculos,Yuca,Yuca consumo en fresco,2022,2022A,350.0,380.0,3800.00,10.00,Transitorio,En fresco,1081001,Manihot esculenta
166729,99,Vichada,99773,Cumaribo,Raíces y tubérculos,Raíces y tubérculos,Yuca,Yuca consumo en fresco,2023,2023A,400.0,350.0,3500.00,10.00,Transitorio,En fresco,1081001,Manihot esculenta
166730,99,Vichada,99773,Cumaribo,Raíces y tubérculos,Raíces y tubérculos,Yuca,Yuca consumo en fresco,2024,2024A,420.0,400.0,4400.00,11.00,Transitorio,En fresco,1081001,Manihot esculenta


----------------------


,codigo_municipio,anio


**Interpretación del resultado:**

_Se crearon las columnas "codigo_municipio" desde Código Dane municipio y "anio" desde Año de EVA. Posteriormente, se identificaron los codigos vacios o mal formados para revision.Tambien se completaron con ceros, los textos numéricos válidos de menor longitud._

_Se crearon combinaciones distintas de código y año y se realizo un anti-join frente a dimension.Finalmente se guardo el resultado en la tabla "sin_correspondencia" y seguidamente se proyecto mediante un comando y se observa que esta vacia, como se ve en la parte anterior._

## 3. Selección para el indicador · 10 minutos

Convertir `Área cosechada` y `Producción` a `area_ha` y `produccion_t`. Seleccionar primero `Cultivo == "Papa"` y año 2024 en `papa_2024`. De ese conjunto, conservar en `apta` las filas con área positiva y producción no negativa. Contar las filas excluidas del indicador dentro de papa_2024.

📘 **Apoyo:** `pd.to_numeric`, `.eq`, `.gt`, `.ge`, `.loc` y `.copy()`.

✅ **Comprobación:** El total de papa_2024 es igual a las filas aptas más las excluidas del indicador. La ausencia de una medida no se reemplaza por cero.

In [4]:
# Desarrollo de la actividad 3
# Crear papa_2024 y apta.
# Mostrar las exclusiones del indicador y conciliar los conteos.
# Convertir Área cosechada y Producción a numérico
eva["area_ha"] = pd.to_numeric(
    eva["Área cosechada"],
    errors="coerce"
)

eva["produccion_t"] = pd.to_numeric(
    eva["Producción"],
    errors="coerce"
)

# Seleccionar Papa del año 2024
papa_2024 = eva.loc[
    eva["Cultivo"].eq("Papa") &
    eva["Año"].eq(2024)
].copy()

# Filas aptas para el indicador
aptas = (
    papa_2024["area_ha"].gt(0) &
    papa_2024["produccion_t"].ge(0)
)

apta = papa_2024.loc[aptas].copy()

# Contar filas excluidas
excluidas_indicador = (~aptas).sum()

# Comprobación
assert len(papa_2024) == len(apta) + excluidas_indicador

# Mostrar resultados
print("Filas de papa_2024:", len(papa_2024))
print("Filas aptas:", len(apta))
print("Filas excluidas:", excluidas_indicador)

display(apta)

Filas de papa_2024: 831
Filas aptas: 779
Filas excluidas: 52


,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Grupo cultivo,Subgrupo,Cultivo,Desagregación cultivo,Año,Periodo,...,Ciclo del cultivo,Estado físico del cultivo,Código del cultivo,Nombre científico del cultivo,codigo_municipio_original,anio_original,codigo_municipio,anio,area_ha,produccion_t
203,5,Antioquia,5001,Medellín,Raíces y tubérculos,Raíces y tubérculos,Papa,Papa todas las variedades,2024,2024A,...,Transitorio,En fresco,1080701,Solanum tuberosum,5001,2024,05001,2024,42.0,672.0
204,5,Antioquia,5001,Medellín,Raíces y tubérculos,Raíces y tubérculos,Papa,Papa todas las variedades,2024,2024B,...,Transitorio,En fresco,1080701,Solanum tuberosum,5001,2024,05001,2024,38.0,608.0
425,5,Antioquia,5002,Abejorral,Raíces y tubérculos,Raíces y tubérculos,Papa,Papa todas las variedades,2024,2024A,...,Transitorio,En fresco,1080701,Solanum tuberosum,5002,2024,05002,2024,80.0,800.0
426,5,Antioquia,5002,Abejorral,Raíces y tubérculos,Raíces y tubérculos,Papa,Papa todas las variedades,2024,2024B,...,Transitorio,En fresco,1080701,Solanum tuberosum,5002,2024,05002,2024,85.0,850.0
590,5,Antioquia,5004,Abriaquí,Raíces y tubérculos,Raíces y tubérculos,Papa,Papa criolla,2024,2024A,...,Transitorio,En fresco,1080702,Solanum phureja,5004,2024,05004,2024,2.5,7.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158661,76,Valle del Cauca,76834,Tuluá,Raíces y tubérculos,Raíces y tubérculos,Papa,Papa todas las variedades,2024,2024B,...,Transitorio,En fresco,1080701,Solanum tuberosum,76834,2024,76834,2024,240.0,4320.0
161744,85,Casanare,85136,La Salina,Raíces y tubérculos,Raíces y tubérculos,Papa,Papa todas las variedades,2024,2024A,...,Transitorio,En fresco,1080701,Solanum tuberosum,85136,2024,85136,2024,6.0,24.0
164202,86,Putumayo,86749,Sibundoy,Raíces y tubérculos,Raíces y tubérculos,Papa,Papa todas las variedades,2024,2024B,...,Transitorio,En fresco,1080701,Solanum tuberosum,86749,2024,86749,2024,6.0,78.0
164294,86,Putumayo,86755,San Francisco,Raíces y tubérculos,Raíces y tubérculos,Papa,Papa todas las variedades,2024,2024B,...,Transitorio,En fresco,1080701,Solanum tuberosum,86755,2024,86755,2024,2.0,20.0


**Interpretación del resultado:**

_Se hizo la conversion de area_ha y produccion_t, de las columnas designadas. Luego se eligio el cultivo de papa y el año 2024, de este conjunto de filas, se obtuvo que hay 779 que son aptas, 52 que fueron excluidas y el total de las filas son 831. Además, el total de 2024 y las suma de las filas aptas y las excluidas son iguales._

## 4. Rendimiento agregado · 15 minutos

Agrupar apta por código municipal, cultivo, estado físico del cultivo y año. Crear `rendimiento` con cantidad de registros, suma de producción y suma de área cosechada. Calcular `rendimiento_t_ha` dividiendo ambas sumas.

📘 **Apoyo:** `groupby(..., as_index=False, dropna=False)`, `agg`, `size` y `sum`.

✅ **Comprobación:** Cada grupo tiene área total positiva. La suma de los conteos por grupo coincide con len(apta). El indicador se calcula como suma de producción / suma de área, conservando cultivo y estado físico en la agrupación.

In [5]:
# Desarrollo de la actividad 4
# Crear rendimiento con los agregados y rendimiento_t_ha.
# Comprobar denominadores y cantidad de registros agrupados.
# Columnas para agrupar
grupos = [
    "codigo_municipio",
    "Cultivo",
    "Estado físico del cultivo",
    "Año"
]

# Construir tabla de rendimiento
rendimiento = (
    apta.groupby(
        grupos,
        as_index=False,
        dropna=False
    )
    .agg(
        cantidad_registros=("produccion_t", "size"),
        produccion_t=("produccion_t", "sum"),
        area_ha=("area_ha", "sum")
    )
)

# Calcular rendimiento en toneladas por hectárea
rendimiento["rendimiento_t_ha"] = (
    rendimiento["produccion_t"] / rendimiento["area_ha"]
)

# Comprobaciones
assert (rendimiento["area_ha"] > 0).all()

assert rendimiento["cantidad_registros"].sum() == len(apta)

# Mostrar resultado
display(rendimiento)

,codigo_municipio,Cultivo,Estado físico del cultivo,Año,cantidad_registros,produccion_t,area_ha,rendimiento_t_ha
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459
...,...,...,...,...,...,...,...,...
287,76834,Papa,En fresco,2024,2,8570.0,490.0,17.489796
288,85136,Papa,En fresco,2024,1,24.0,6.0,4.000000
289,86749,Papa,En fresco,2024,1,78.0,6.0,13.000000
290,86755,Papa,En fresco,2024,1,20.0,2.0,10.000000


**Interpretación del resultado:**

_A la validacion de fila apta de punto anterior, se agrupo por código municipal, cultivo, estado físico del cultivo y año. Posteriormente, se  creo la tabla rendimiento con las columnas "cantidad de registros", "suma de producción" y "suma de área cosechada". Por otro lado, se  calcula la columna "rendimiento_t_ha" dividiendo suma de produccion y suma de área cosechada. Tambien se pude observar que cada grupo tiene área total positiva y que el grupo conicide con el valor que saca el comando "len(apta)"._ 

## 5. Unión con nombres y clasificación · 10 minutos

Relacionar rendimiento con dimension mediante un left merge por codigo_municipio. Declarar `validate="many_to_one"` e incorporar un indicador de correspondencia. Guardar el resultado en `rendimiento_municipal` y mostrar los códigos sin nombre territorial, si existen.

📘 **Apoyo:** `merge`, `validate`, `indicator` y `len`.

✅ **Comprobación:** La cantidad de filas antes y después de la unión debe ser idéntica. Un fallo de cardinalidad debe investigarse; no se resuelve eliminando arbitrariamente claves de la dimensión.

In [6]:
# Desarrollo de la actividad 5
# Unir rendimiento con dimension.
# Verificar la conservación de filas y presentar la cobertura de la unión.
# Guardar la cantidad de filas antes de la unión
filas_antes = len(rendimiento)

# Relacionar rendimiento con dimension
rendimiento_municipal = rendimiento.merge(
    dimension,
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Comprobar que no se hayan perdido ni agregado filas
filas_despues = len(rendimiento_municipal)

assert filas_antes == filas_despues

# Identificar códigos sin correspondencia territorial
sin_nombre_territorial = rendimiento_municipal.loc[
    rendimiento_municipal["_merge"].eq("left_only"),
    ["codigo_municipio", "Cultivo", "Año", "_merge"]
].drop_duplicates()

# Mostrar resultados
print("Filas antes de la unión:", filas_antes)
print("Filas después de la unión:", filas_despues)
print("Códigos sin correspondencia:", len(sin_nombre_territorial))
print("----------------------------")
display(rendimiento_municipal)
print("----------------------------")
display(sin_nombre_territorial)

Filas antes de la unión: 292
Filas después de la unión: 292
Códigos sin correspondencia: 0
----------------------------


,codigo_municipio,Cultivo,Estado físico del cultivo,Año,cantidad_registros,produccion_t,area_ha,rendimiento_t_ha,nombre_departamental,nombre_municipal,clasificacion_territorial,_merge
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000,ANTIOQUIA,MEDELLÍN,Municipio,both
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000,ANTIOQUIA,ABEJORRAL,Municipio,both
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000,ANTIOQUIA,ABRIAQUÍ,Municipio,both
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353,ANTIOQUIA,BARBOSA,Municipio,both
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459,ANTIOQUIA,BELMIRA,Municipio,both
...,...,...,...,...,...,...,...,...,...,...,...,...
287,76834,Papa,En fresco,2024,2,8570.0,490.0,17.489796,VALLE DEL CAUCA,TULUÁ,Municipio,both
288,85136,Papa,En fresco,2024,1,24.0,6.0,4.000000,CASANARE,LA SALINA,Municipio,both
289,86749,Papa,En fresco,2024,1,78.0,6.0,13.000000,PUTUMAYO,SIBUNDOY,Municipio,both
290,86755,Papa,En fresco,2024,1,20.0,2.0,10.000000,PUTUMAYO,SAN FRANCISCO,Municipio,both


----------------------------


,codigo_municipio,Cultivo,Año,_merge


**Interpretación del resultado:**

_Se relacionaron rendimiento con dimension mediante un left merge por codigo_municipio y el resultado se guardo en rendimiento_municipal. Por otro lado se verifico que la cantidad de filas antes y despues de la unión es casi la misma o identica. Ademas, no se encontraron codigos sin nombre territorial con se ve en la ultima tabla que esta antes._

## 6. Exportación e interpretación · 10 minutos

Exportar rendimiento_municipal a `OUT / "rendimiento_municipal.csv"` y sin_correspondencia a `OUT / "codigos_sin_correspondencia.csv"`, ambos sin índice. Explicar por qué se dividen sumas en lugar de promediar rendimientos individuales y qué limitación introduce comparar EVA con un catálogo territorial de corte 2024.

📘 **Apoyo:** `to_csv` y los resultados de las actividades anteriores.

✅ **Comprobación:** El archivo de códigos conserva sus cabeceras aunque no tenga filas. La explicación distingue una coincidencia de códigos de una equivalencia histórica de límites.

In [7]:
# Desarrollo de la actividad 6
# Exportar las dos tablas.
# Explicar el denominador y la limitación temporal en la celda siguiente.
# Exportar rendimiento municipal
from pathlib import Path

# Crear carpeta de salida
OUT = Path("OUT")
OUT.mkdir(exist_ok=True)

# Exportar rendimiento municipal
rendimiento_municipal.to_csv(
    OUT / "rendimiento_municipal.csv",
    index=False
)

# Exportar códigos sin correspondencia
sin_correspondencia.to_csv(
    OUT / "codigos_sin_correspondencia.csv",
    index=False
)

print("Filas de rendimiento municipal:", len(rendimiento_municipal))
print("Filas sin correspondencia:", len(sin_correspondencia))

print(OUT / "rendimiento_municipal.csv")
print(OUT / "codigos_sin_correspondencia.csv")

Filas de rendimiento municipal: 292
Filas sin correspondencia: 0
OUT/rendimiento_municipal.csv
OUT/codigos_sin_correspondencia.csv


**Interpretación del resultado:**

_Se exporto rendimiento_municipal a OUT / "rendimiento_municipal.csv" y sin_correspondencia a OUT / "codigos_sin_correspondencia.csv"._

_ Se divide la suma de la producción entre la suma del área cosechada porque así se obtiene el rendimiento global del grupo en toneladas por hectárea, teniendo en cuenta las diferencias en la superficie de cada registro. Porque si promediaramos solo los rendimientos individuales, se podría dar el mismo peso a registros con áreas diferentes y producir un resultado que no representa el rendimiento conjunto del cultivo._

_Comparar EVA con un catálogo territorial de corte 2024, solo permite observar los valores que este relacionados con ese corte y esto limita saber  que cambios pudieron haberse desarrollado en comparacion con los años anteriores, de manera positivo o negativa._

## Entrega

El notebook completado se conserva como `soluciones/PD03_integracion_territorial.ipynb`. Las salidas regenerables se guardan en `kit/salidas/soluciones/PD03/`.

✅ Deben quedar visibles el control de la dimensión, el anti-join, el conteo de exclusiones, la conciliación de grupos y la conservación de filas en el merge.

⚠️ Una coincidencia de códigos no prueba estabilidad de límites municipales a través del tiempo. El rendimiento agregado se interpreta para el cultivo y estado físico seleccionados; no debe sumarse con rendimientos de otros cultivos.